# Token-temperature scan — tinker notebook

Wraps the functions from `token_temperature_scan.py` + `plot_T_scan_comparison.py` so you can load a checkpoint once, run partial scans, and re-plot without restarting.

Requires the `hep4m2` conda env (for PflowMetrics). Run on a GPU node.

**Structure:**
1. Setup + imports
2. Load checkpoint + val dataset + PflowMetrics  (do this **once**, keep in memory)
3. Run scans — edit `T_grid` and re-run to tinker
4. Plot nano-hep T-sweep curves
5. Overlay with HEP4M CSV (if present)
6. Diagnostic cells: N_pred vs N_true, pflow event-level distributions, etc.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys, importlib, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO = Path('/global/cfs/cdirs/m4958/usr/danieltm/Side_Work/FoundationModels/nano-hep')
HEP4M = '/global/cfs/cdirs/m4958/usr/danieltm/Side_Work/FoundationModels/HEP4M'
for p in (str(REPO), HEP4M, str(REPO / 'notebooks')):
    if p not in sys.path:
        sys.path.insert(0, p)

import token_temperature_scan as tts
import plot_T_scan_comparison as pcmp
importlib.reload(tts); importlib.reload(pcmp)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '|', torch.cuda.get_device_name(0) if device == 'cuda' else '-')

## 2. Load checkpoint + val dataset + PflowMetrics

Cache these as globals; subsequent cells reuse them. Re-run this cell only if you want to swap checkpoints.

In [ ]:
CKPT = Path('/global/cfs/cdirs/m4958/usr/danieltm/Side_Work/FoundationModels/nano-hep/runs/nano_hep_89M_ddp/last.ckpt')
N_EVENTS = 256   # tinker knob for all scans below

print(f'loading ckpt @ {CKPT.stat().st_mtime}')
model, vocab, cfg, step = tts.load_model_from_ckpt(CKPT, device=device)
val_ds = tts.build_val_ds(cfg, vocab, max_events=N_EVENTS)
pflow = tts.PflowMetrics(
    modality_dict_path=cfg['pflow_metrics']['modality_dict_path'],
    output_modality=cfg['data']['output_modality'],
    device=device,
)
print(f'step={step}  block_size={val_ds.block_size}  vocab.total={vocab.total}')

## 3. Run the T-scan

Edit `T_GRID` to skip / extend / add temperatures. Each `run_one_T` call is ~2 min per T at `N_EVENTS=256`. Results persist as JSON after each T so you can crash-recover.

In [ ]:
T_GRID = [0.5, 0.7, 1.0, 1.3]   # tinker here; start small, add more if interesting
OUT = REPO / 'notebooks' / 'results_T_scan_nb'
OUT.mkdir(parents=True, exist_ok=True)

rows = []
for T in T_GRID:
    print(f'--- T = {T} ---', flush=True)
    res = tts.run_one_T(
        model, vocab, val_ds, T, N_EVENTS, device, pflow, OUT,
        argmax_at_T1=True, seed=42,
    )
    for k in ('cardinality_acc', 'cardinality_mae',
              'pflow_median_jet_pt_response', 'pflow_iqr_jet_pt_response',
              'pflow_mean_reco_cardinality_at_threshold'):
        if k in res:
            print(f'  {k}: {res[k]:.4f}')
    rows.append(res)

# Stash raw dicts and a scalar DataFrame
df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in rows])
df.to_csv(OUT / 'nano_hep_T_scan.csv', index=False)
with open(OUT / 'nano_hep_T_scan.json', 'w') as f:
    json.dump(rows, f, indent=2)
df

## 4. Plot nano-hep T-sweep curves

Self-contained — re-run after you add more T values to `rows`.

In [ ]:
df = pd.read_csv(OUT / 'nano_hep_T_scan.csv').sort_values('T').reset_index(drop=True)
truth_n = float(df['n_true_mean'].iloc[0])

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
pcmp.plot_metric if False else None  # silence lint

def plot_single(ax, y, ylabel, title, truth=None):
    ax.plot(df['T'], y, 'o-', color='#1f77b4', lw=2, ms=7, label='nano-hep')
    if truth is not None:
        ax.axhline(truth, color='k', ls=':', lw=1, alpha=0.6, label=f'truth ({truth:.2f})')
    ax.axvline(1.0, color='grey', ls='--', lw=0.8, alpha=0.5)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(True, alpha=0.3)

plot_single(axes[0,0], df['pflow_median_jet_pt_response'], 'median jet pT response', 'Jet pT response: median', truth=1.0)
plot_single(axes[0,1], df['pflow_iqr_jet_pt_response'], 'IQR(jet pT response)', 'Jet pT response: IQR (lower=tighter)')
plot_single(axes[0,2], df['pflow_mean_reco_cardinality_at_threshold'], 'mean N reco', 'Reco cardinality', truth=truth_n)
plot_single(axes[1,0], df['cardinality_acc'], 'P(n_pred == n_true)', 'Cardinality accuracy')
plot_single(axes[1,1], df['cardinality_mae'], '|n_pred - n_true|', 'Cardinality MAE (lower=better)')
plot_single(axes[1,2], df['n_pred_mean'], 'mean n_pred', 'Predicted N', truth=truth_n)
axes[0,0].legend(loc='best', fontsize=9)
for ax in axes[1,:]: ax.set_xlabel('Temperature T')
fig.suptitle(f'nano-hep T-scan (ckpt step {step}, N={N_EVENTS} events)', fontsize=12)
fig.tight_layout()

## 5. Overlay with HEP4M (if CSV exists)

In [ ]:
HEP4M_CSV = Path(HEP4M) / 'notebooks/paper/results_T_scan/hep4m_T_scan.csv'
if HEP4M_CSV.exists():
    hep4m = pd.read_csv(HEP4M_CSV).sort_values('T').reset_index(drop=True)
    fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True)
    def overlay(ax, col, ylabel, title, truth=None):
        ax.plot(df['T'], df[col], 'o-', color='#1f77b4', lw=2, ms=7, label=f'nano-hep (step {step})')
        ax.plot(hep4m['T'], hep4m[col], 's-', color='#d62728', lw=2, ms=7, label='HEP4M')
        if truth is not None:
            ax.axhline(truth, color='k', ls=':', lw=1, alpha=0.6, label=f'truth ({truth:.2f})')
        ax.axvline(1.0, color='grey', ls='--', lw=0.8, alpha=0.5)
        ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(True, alpha=0.3)
    overlay(axes[0,0], 'pflow_median_jet_pt_response', 'median jet pT', 'Jet pT: median', truth=1.0)
    overlay(axes[0,1], 'pflow_iqr_jet_pt_response', 'IQR', 'Jet pT: IQR')
    overlay(axes[0,2], 'pflow_mean_reco_cardinality_at_threshold', 'mean N reco', 'Reco N', truth=truth_n)
    overlay(axes[1,0], 'cardinality_acc', 'exact-N', 'Cardinality accuracy')
    overlay(axes[1,1], 'cardinality_mae', 'MAE', 'Cardinality MAE')
    overlay(axes[1,2], 'n_pred_mean', 'mean n_pred', 'Predicted N', truth=truth_n)
    axes[0,0].legend(loc='best', fontsize=9)
    for ax in axes[1,:]: ax.set_xlabel('Temperature T')
    fig.tight_layout()
else:
    print(f'HEP4M CSV missing: {HEP4M_CSV}')

## 6. Tinker cell — single-T diagnostic

Quick re-run for one T + inspect per-event arrays. Good for isolating an outlier or trying a non-grid temperature.

In [ ]:
T_PLAY = 0.9
res_play = tts.run_one_T(
    model, vocab, val_ds, T_PLAY, N_EVENTS, device, pflow, OUT,
    argmax_at_T1=False, seed=42,
)
print({k: v for k, v in res_play.items() if not k.startswith('_')})

# Per-event counts
n_pred = np.array(res_play['_n_pred_arr'])
n_true = np.array(res_play['_n_true_arr'])
resid = n_pred - n_true

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(resid, bins=range(resid.min()-1, resid.max()+2), alpha=0.7, edgecolor='k')
ax[0].axvline(0, color='r', ls='--', lw=1)
ax[0].set_xlabel('n_pred - n_true'); ax[0].set_title(f'T={T_PLAY}: residual N'); ax[0].grid(alpha=0.3)
ax[1].scatter(n_true, n_pred, alpha=0.5, s=20)
lim = max(n_true.max(), n_pred.max()) + 1
ax[1].plot([0, lim], [0, lim], 'k--', alpha=0.5)
ax[1].set_xlabel('n_true'); ax[1].set_ylabel('n_pred'); ax[1].set_title('predicted vs truth'); ax[1].grid(alpha=0.3)
fig.tight_layout()

## 7. Free-form — whatever else you want to poke at

E.g. per-T reco/truth histograms, jet-pt-response violin, EOS-probability over the generation, etc. The globals `model`, `vocab`, `val_ds`, `pflow` are all live here.

In [ ]:
# Example: emit raw tokens for a single event at T=1 argmax and inspect the parse
EVENT_IDX = 0
pc, pp = tts.ar_decode_tempered(
    model, vocab, val_ds, EVENT_IDX, T=1.0,
    max_new_tokens=val_ds.block_size, device=device,
    argmax_at_T1=True,
)
print(f'event {EVENT_IDX}: {pc.shape[0]} pred particles (truth has '
      f'{int(val_ds.out_mm.offsets[EVENT_IDX+1] - val_ds.out_mm.offsets[EVENT_IDX])})')
print('pred content codes  (shape):', pc.shape)
print('pred pos codes      (shape):', pp.shape)